# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JasperOwen/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

import duckdb
from huggingface_hub import login

login(token=hf_token)

con = duckdb.connect()
con.sql("SET enable_http_metadata_cache=true;")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
from huggingface_hub import hf_hub_download

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=hf_token
)

file_path

'/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet'

In [3]:
feature_frame = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(CASE WHEN report_date <= '2026-03-15' THEN sessions_ai ELSE 0 END) AS sessions_ai,

    SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_clicks ELSE 0 END) AS early_march_clicks,

    AVG(CASE WHEN report_date <= '2026-03-15' THEN gsc_avg_position ELSE NULL END) AS early_march_avg_position,

    SUM(CASE WHEN report_date <= '2026-03-15' THEN ga4_engaged_sessions ELSE 0 END) AS ga4_engaged_sessions,

    AVG(CASE WHEN report_date <= '2026-03-15' THEN ga4_total_engagement_sec ELSE 0 END) AS early_march_engagement_sec,

FROM read_parquet('{file_path}')
WHERE gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

feature_frame["early_march_avg_position"] = feature_frame["early_march_avg_position"].fillna(999)

feature_frame.head()

,client_hash_id,content_hash_id,sessions_ai,early_march_clicks,early_march_avg_position,ga4_engaged_sessions,early_march_engagement_sec
0,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,2.0,2.0,4.095154,0.0,0.222222
1,client_65de48885f4ef01b,content_e25ea7297a1dffd3,1.0,10.0,4.319753,1.0,2.840000
2,client_65de48885f4ef01b,content_3c286ded8bd68120,0.0,7.0,8.777106,1.0,1.000000
3,client_65de48885f4ef01b,content_b2108e8fe3360fa6,1.0,1.0,5.475501,0.0,0.000000
4,client_65de48885f4ef01b,content_ff867882e604fa96,1.0,0.0,2.850000,0.0,0.000000


In [4]:
X = feature_frame[
    [
        "sessions_ai",
        "early_march_clicks",
        "early_march_avg_position",
        "ga4_engaged_sessions",
        "early_march_engagement_sec"
    ]
].copy()

X.head()

,sessions_ai,early_march_clicks,early_march_avg_position,ga4_engaged_sessions,early_march_engagement_sec
0,2.0,2.0,4.095154,0.0,0.222222
1,1.0,10.0,4.319753,1.0,2.840000
2,0.0,7.0,8.777106,1.0,1.000000
3,1.0,1.0,5.475501,0.0,0.000000
4,1.0,0.0,2.850000,0.0,0.000000


In [5]:
X.isna().sum()

,0
sessions_ai,0
early_march_clicks,0
early_march_avg_position,0
ga4_engaged_sessions,0
early_march_engagement_sec,0


In [6]:
X.isna().sum()

,0
sessions_ai,0
early_march_clicks,0
early_march_avg_position,0
ga4_engaged_sessions,0
early_march_engagement_sec,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

`sessions_ai`
- Meaning: Number of times the page was accessed using a link provided by AI
- Missing handling: No missing values were present to handle.
- Categorical handling: Used as a numerical feature.
- Available when: Available before the prediction moment because AI referrals up to 15th March 2026 are historical data.

`early_march_clicks`
- Meaning: Number of times the page was accessed using a link found in a google search
- Missing handling: No missing values were present to handle.
- Categorical handling: Used as a numerical feature.
- Available when: Available before the prediction moment because gsc clicks up to 15th March 2026 are avaliable as historical data.

`early_march_avg_position`
- Meaning: Average Google Search ranking position before the decision moment.
- Missing handling: Missing values were replaced with 999 because pages without Search Console ranking data do not have a valid position. Also, low numbers indicate that the page was ranked highly in a google search so a large number was needed.
- Categorical handling: Used as a numerical feature.
- Available when: Available before the prediction moment because gsc data avaliable up to 15th March 2026 is historical data.

`ga4_engaged_sessions`
- Meaning: Number of times a visitor engaged with the features provided in a page
- Missing handling: No missing values were present to handle.
- Categorical handling: Used as a numerical feature.
- Available when: Available before the prediction moment because ga4 data avaliable up to 15th March 2026 is historical data.

`early_march_engagement_sec`
- Meaning: On average how long a visitor spent looking at a page
- Missing handling: No missing values were present to handle.
- Categorical handling: Used as a numerical feature.
- Available when: Available before the prediction moment because ga4 data avaliable up to 15th March 2026 is historical data.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
future = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_clicks ELSE 0 END) AS late_march_clicks

FROM read_parquet('{file_path}')
WHERE gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

future_frame = feature_frame.merge(
    future,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

In [8]:
future_frame['march_click_change'] = future_frame['late_march_clicks'] - future_frame['early_march_clicks']
future_frame.head(5)

,client_hash_id,content_hash_id,sessions_ai,early_march_clicks,early_march_avg_position,ga4_engaged_sessions,early_march_engagement_sec,late_march_clicks,march_click_change
0,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,2.0,2.0,4.095154,0.0,0.222222,0.0,-2.0
1,client_65de48885f4ef01b,content_e25ea7297a1dffd3,1.0,10.0,4.319753,1.0,2.840000,13.0,3.0
2,client_65de48885f4ef01b,content_3c286ded8bd68120,0.0,7.0,8.777106,1.0,1.000000,8.0,1.0
3,client_65de48885f4ef01b,content_b2108e8fe3360fa6,1.0,1.0,5.475501,0.0,0.000000,7.0,6.0
4,client_65de48885f4ef01b,content_ff867882e604fa96,1.0,0.0,2.850000,0.0,0.000000,0.0,0.0


In [9]:
outcome_frame = future_frame[
    [
        "client_hash_id",
        "content_hash_id",
        "march_click_change"
    ]
].copy()

outcome_frame.head()

,client_hash_id,content_hash_id,march_click_change
0,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,-2.0
1,client_65de48885f4ef01b,content_e25ea7297a1dffd3,3.0
2,client_65de48885f4ef01b,content_3c286ded8bd68120,1.0
3,client_65de48885f4ef01b,content_b2108e8fe3360fa6,6.0
4,client_65de48885f4ef01b,content_ff867882e604fa96,0.0


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

clean_features = [
    "sessions_ai",
    "early_march_clicks",
    "early_march_avg_position",
    "ga4_engaged_sessions",
    "early_march_engagement_sec"
]

training_frame = feature_frame.merge(
    outcome_frame[
        [
            "client_hash_id",
            "content_hash_id",
            "march_click_change"
        ]
    ],
    on=[
        "client_hash_id",
        "content_hash_id"
    ]
)

X_clean = training_frame[clean_features]
y_clean = training_frame["march_click_change"]

X_train, X_test, y_train, y_test = train_test_split(
    X_clean,
    y_clean,
    test_size=0.2,
    random_state=42
)

clean_model = LinearRegression()

clean_model.fit(
    X_train,
    y_train
)

clean_predictions = clean_model.predict(X_test)

clean_score = r2_score(
    y_test,
    clean_predictions
)

clean_score

0.14321815297369622

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

leaky_features = [
    "sessions_ai",
    "early_march_clicks",
    "early_march_avg_position",
    "ga4_engaged_sessions",
    "early_march_engagement_sec",
    "late_march_clicks"
]

training_frame = future_frame.copy()

X_leaky = training_frame[leaky_features]
y_leaky = training_frame["march_click_change"]

X_train, X_test, y_train, y_test = train_test_split(
    X_leaky,
    y_leaky,
    test_size=0.2,
    random_state=42
)

leaky_model = LinearRegression()

leaky_model.fit(
    X_train,
    y_train
)

leaky_predictions = leaky_model.predict(X_test)

leaky_score = r2_score(
    y_test,
    leaky_predictions
)

leaky_score

1.0

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Column | Why it was excluded |
| ------ | -------------------- |
| `post March 15th data`| `causes data leakage if used`  |
| `month` | `redundant in favour of report-date` |
| `client_has_gsc` | `not needed as row-level sql gsc_data_available IS TRUE filters more accurately` |
| `client_has_ga4` |  `not needed as row-level sql ga4_data_available IS TRUE filters more accurately` |
| `ai_chatgpt` | `verified to be redundant with sessions_ai.  Aggregation is more useful than any single platform` |
| `ai_perplexity` | `verified to be redundant with sessions_ai.  Aggregation is more useful than any single platform` |
| `ai_gemini` | `verified to be redundant with sessions_ai.  Aggregation is more useful than any single platform` |
| `ai_copilot` | `verified to be redundant with sessions_ai.  Aggregation is more useful than any single platform` |
| `ai_claude` | `verified to be redundant with sessions_ai.  Aggregation is more useful than any single platform` |
| `ai_meta` | `verified to be redundant with sessions_ai.  Aggregation is more useful than any single platform` |
| `ai_other` | `verified to be redundant with sessions_ai.  Aggregation is more useful than any single platform` |
| `sessions_organic` | `excluded due to 5-feature budget in favour of sessions_ai to account for ai referrals`  |
| `sessions_direct` | `excluded due to 5-feature budget in favour of sessions_ai to account for ai referrals` |
| `sessions_referral` | `excluded due to 5-feature budget in favour of sessions_ai to account for ai referrals` |
| `sessions_social` | `excluded due to 5-feature budget in favour of sessions_ai to account for ai referrals` |
| `sessions_paid` | `excluded due to 5-feature budget in favour of sessions_ai to account for ai referrals` |
| `ga4_sessions` | `excluded due to 5-feature budget in favour of engaged_sessions to prove content usefulness` |
| `ga4_pageviews` | `excluded due to 5-feature budget in favour of engaged_sessions to prove content usefulness` |
| `scroll_events` | `excluded due to 5-feature budget in favour of engaged_sessions to prove content usefulness` |
| `gsc_impressions` | `excluded due to 5-feature budget in favour of average position` |
| `gsc_sum_position` | `excluded due to 5-feature budget in favour of average position` |

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
    COUNT(*) AS mismatch_count
FROM read_parquet('{file_path}')
WHERE sessions_ai != (
    ai_chatgpt + ai_perplexity + ai_gemini + ai_copilot + ai_claude + ai_meta + ai_other
)
""").df()

,mismatch_count
0,2


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.